# fwd 3d

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/main/examples/forward/fwd-3d.ipynb)

> Select **Runtime → Change runtime type → GPU or TPU** in Colab to demonstrate the hardware-acceleration story.


In [ ]:
# Install beamax for Google Colab. Safe to skip when running locally.
%%capture
%pip install --quiet "beamax[kwave] @ git+https://github.com/elma16/beamax.git"


In [ ]:
#!/usr/bin/env python
# coding: utf-8

import jax
import jax.numpy as jnp
import numpy as np
from pathlib import Path
from time import time

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1 import make_axes_locatable

from beamax import geometry, utils, plotter
from beamax.decomposition import DyadicDecomposition
from beamax.transforms import MSWPT
from beamax.gb import gb_solvers
from beamax.solvers import KWaveSolver, MSGBSolver, HybridSolver, ShardingStrategy

from kwave.options.simulation_execution_options import SimulationExecutionOptions
from kwave.options.simulation_options import SimulationOptions

# ============================================================================
# CONFIGURATION
# ============================================================================
jax.config.update("jax_enable_x64", False)

ROOT_DIR = utils.detect_root()
PLOT_DIR = Path(ROOT_DIR / "plots")
DATA_DIR = Path(ROOT_DIR / "data")
PLOT_DIR.mkdir(exist_ok=True)
pltgb = plotter.PlotHelper()


# ============================================================================
# DOMAIN SETUP
# ============================================================================
d = 3
N = (64,) * d
dx = (1e-4,) * d
periodic = (True,) * d
box_aspect_ratio = (1,) * d
num_levels = 2
num_boxes_levels = tuple([2 ** (i + 2) for i in range(num_levels)])


def c_fn(x):
    return 1500 + 0 * x[..., 0]


cfl = (jnp.sqrt(3) / 4).round(3)
domain = geometry.Domain(N=N, dx=dx, c=c_fn, cfl=cfl, periodic=periodic)
ts = domain.generate_time_domain()

print(f"Grid: {N}, dx={dx}")
print(f"Time steps: {len(ts)}, dt={float(ts[1] - ts[0]):.3e}s")

# ============================================================================
# LOAD DATA (OA-BREAST)
# ============================================================================
phantom_path = DATA_DIR / "NumericalBreastPhantoms-selected/hdf5/Neg_07_Left.h5"

p0, c_3d, meta3d = utils.load_oabreast_p0_c(
    phantom_path,
    dim="3d",
    axis_order="ZYX",
    source_spacing_mm=dx,
    target_shape=N,
    normalize_p0=False,
)

# Update c function
# c_fn = utils.make_c_function_from_grid(c_3d, dx, (0, 0, 0))
domain = geometry.Domain(N=N, dx=dx, c=c_fn, cfl=cfl, periodic=periodic)

p0 = p0.real
print(f"p0 range: [{p0.min():.2e}, {p0.max():.2e}]")

# ============================================================================
# SENSORS (planar array at z=0)
# ============================================================================
binary_mask = jnp.zeros(N).at[..., 0].set(1)
sensors = geometry.Sensor(binary_mask=binary_mask, domain=domain)
num_sensors = int(binary_mask.sum())
print(f"Sensors: {num_sensors} points")

# ============================================================================
# MSWPT & COEFFICIENT THRESHOLDING
# ============================================================================
windowing = "rectangular_mirror"
redundancy = 2

dyadic_decomp = DyadicDecomposition(num_levels, N, num_boxes_levels, box_aspect_ratio)
wpt = MSWPT(dyadic_decomp, redundancy, windowing)
wpt_none = MSWPT(dyadic_decomp, redundancy, "none")

coeffs = wpt.forward(p0, input_type="spatial")
coeffs_array = wpt.convert_to_array(coeffs)

# Adaptive thresholding
tau = 0.01
K = utils.choose_K_by_tau(
    coeffs, p0, wpt_none, dyadic_decomp, wpt, tau=tau, Kmin=512, Kmax=None, num_steps=10
)
print(f"Chosen K={K} for τ={tau:.1%}")

indices, coeff_vals = utils.select_levelaware_topK_indices(
    coeffs, dyadic_decomp, wpt, K
)
# threshold = int(indices.size)
threshold = 100

# Reconstruct from thresholded coefficients
thresholded_coeffs = jnp.zeros_like(coeffs).at[indices].set(coeff_vals)
data_recon = wpt_none.inverse(thresholded_coeffs, output_type="spatial")

# ============================================================================
# SOLVERS
# ============================================================================
batch_size = 100
sum_method = "scan_real"
strategy = "top_n"

# MSGB
num_devices = jax.device_count()
mesh = jax.make_mesh((num_devices,), ("x",))
sharding_strategy = ShardingStrategy(mesh, beam_axis="x")

msgb_solver = MSGBSolver(
    thr=threshold,
    thr_strat=strategy,
    batch_size=batch_size,
    input_type="spatial",
    ode_solver=gb_solvers.solve_hom_diag,
    sum_method=sum_method,
    sharding=sharding_strategy,
)

# k-Wave
simulation_options = SimulationOptions(
    data_cast="double",
    smooth_p0=False,
    save_to_disk=True,
)
execution_options = SimulationExecutionOptions(
    is_gpu_simulation=False, delete_data=False, verbose_level=0, show_sim_log=False
)
kwave_solver = KWaveSolver(simulation_options, execution_options)

# Hybrid
cutoff_freq = 16
hybrid_solver = HybridSolver(
    lf_solver=kwave_solver,
    hf_solver=msgb_solver,
    downsample=False,
    cutoff_freq=cutoff_freq,
    input_type="spatial",
    interp_method="fourier",
    dt_oversample=0,
    beta=12.0,
)

# ============================================================================
# RUN FORWARD SIMULATIONS
# ============================================================================
print("\n" + "=" * 60)
print("RUNNING FORWARD SIMULATIONS")
print("=" * 60)

# MSGB
print("\nMSGB...")
t1 = time()
gb_data = msgb_solver.forward(p0, domain, sensors, ts, wpt)[0].block_until_ready()
gb_data = gb_data.reshape(len(ts), N[0], N[1]).transpose(0, 2, 1)
t_msgb = time() - t1
print(f"  Time: {t_msgb:.2f}s")

# k-Wave
print("\nk-Wave...")
t1 = time()
kw_data = kwave_solver.forward(p0, domain, sensors.binary_mask, ts)
kw_data = kw_data.reshape(len(ts), N[0], N[1]).transpose(0, 2, 1)
t_kwave = time() - t1
print(f"  Time: {t_kwave:.2f}s")

# Hybrid
print("\nHybrid...")
t1 = time()
hyb_data = hybrid_solver.forward(p0, domain, sensors, ts, wpt)
t_hybrid = time() - t1
print(f"  Time: {t_hybrid:.2f}s")

# Error metrics
print("\n" + "=" * 60)
print("ERROR METRICS")
print("=" * 60)
err_msgb = jnp.linalg.norm(gb_data - kw_data) / jnp.linalg.norm(kw_data)
err_hyb = jnp.linalg.norm(hyb_data - kw_data) / jnp.linalg.norm(kw_data)
print(f"MSGB  rel L2 error: {100 * err_msgb:.2f}%")
print(f"Hybrid rel L2 error: {100 * err_hyb:.2f}%")


exp = 1

In [ ]:
pltgb = plotter.PlotHelper()

# Convert to numpy
_p0 = np.asarray(p0.real)
_coeffs = np.asarray(coeffs_array)
_recon_diff = np.asarray(data_recon.real - p0)

# MIPs over z for p0 and error
p0_xy_mip = np.max(_p0, axis=2)  # (Nx, Ny)
err_xy_mip = np.max(np.abs(_recon_diff), axis=2)

# For coefficients: take XY slice at middle kz
coeff_xy = np.abs(_coeffs[:, :, _coeffs.shape[2] // 2])

# Shared norms
p0_norm = mcolors.Normalize(vmin=float(p0_xy_mip.min()), vmax=float(p0_xy_mip.max()))
err_norm = mcolors.Normalize(vmin=float(err_xy_mip.min()), vmax=float(err_xy_mip.max()))

fig1 = plt.figure(figsize=(14, 4))
gs1 = fig1.add_gridspec(nrows=1, ncols=3, wspace=0.35)

# Panel 1: p0 MIP (XY)
ax1 = fig1.add_subplot(gs1[0, 0])
im1 = ax1.imshow(
    p0_xy_mip.T,
    origin="lower",
    cmap="hot",
    norm=p0_norm,
    extent=[0, N[0] * dx[0], 0, N[1] * dx[1]],
)
ax1.set_title(r"$\max_z\, p_0(\mathbf{x})$")
ax1.set_xlabel("x (m)")
ax1.set_ylabel("y (m)")
ax1.set_xticks([])
ax1.set_yticks([])
div1 = make_axes_locatable(ax1)
cax1 = div1.append_axes("right", size="5%", pad=0.05)
fig1.colorbar(im1, cax=cax1)

# Panel 2: MSWPT coefficients (kx,ky) with boxes
ax2 = fig1.add_subplot(gs1[0, 1])
extent_k = [-N[0] // 2, N[0] // 2, -N[1] // 2, N[1] // 2]
pltgb.plot_coeffs_with_boxes(
    coeff_xy,
    dyadic_decomp,
    ax=ax2,
    plane="xy",
    slice_index=_coeffs.shape[2] // 2,
    extent=extent_k,
    title=r"$|c_{\ell,j,k}|$ (slice $k_z$ mid)",
    show_colorbar=True,
)
ax2.set_xlabel(r"$k_x$")
ax2.set_ylabel(r"$k_y$")
ax2.set_xticks([])
ax2.set_yticks([])

# Panel 3: reconstruction error MIP
ax3 = fig1.add_subplot(gs1[0, 2])
im3 = ax3.imshow(
    err_xy_mip.T,
    origin="lower",
    cmap="hot",
    norm=err_norm,
    extent=[0, N[0] * dx[0], 0, N[1] * dx[1]],
)
ax3.set_title(r"$\max_z\,|p_0^{\mathrm{recon}} - p_0|$")
ax3.set_xlabel("x (m)")
ax3.set_ylabel("y (m)")
ax3.set_xticks([])
ax3.set_yticks([])
div3 = make_axes_locatable(ax3)
cax3 = div3.append_axes("right", size="5%", pad=0.05)
fig1.colorbar(im3, cax=cax3)

fig1.tight_layout()
fig1.savefig(PLOT_DIR / f"3d_fig1_initial_{exp}.png", dpi=300, bbox_inches="tight")
plt.close(fig1)

In [ ]:
# Assume you've already run the solvers and got:
# kw_data   : (Nt, Ny, Nz)
# gb_data   : (Nt, Ny, Nz)
# hybrid_data : (Nt, Ny, Nz)
# ts        : (Nt,)

hybrid_data = gb_data

_kw3 = np.asarray(kw_data)
_msgb3 = np.asarray(gb_data)
_hyb3 = np.asarray(hybrid_data)
_ts = np.asarray(ts)

# MIP over z to get (Nt, Ny)
_kw = np.max(_kw3, axis=2)
_msgb = np.max(_msgb3, axis=2)
_hyb = np.max(_hyb3, axis=2)

# Differences
_d_kw_msgb = _kw - _msgb
_d_kw_hyb = _kw - _hyb

# Normalisations
s_min = float(min(_kw.min(), _msgb.min(), _hyb.min()))
s_max = float(max(_kw.max(), _msgb.max(), _hyb.max()))
s_norm = mcolors.Normalize(vmin=s_min, vmax=s_max)

d_max = float(max(np.max(np.abs(_d_kw_msgb)), np.max(np.abs(_d_kw_hyb))))
d_norm = mcolors.Normalize(vmin=-d_max, vmax=d_max)

Ny = _kw.shape[1]
extent = [0, Ny * dx[1], float(_ts.max()), float(_ts.min())]  # y_s vs t

# Choose a profile at mid y
y_idx = Ny // 2
y_pos = (Ny * dx[1]) * (y_idx / Ny)

fig2 = plt.figure(figsize=(10, 7))
gs2 = fig2.add_gridspec(
    nrows=2, ncols=3, height_ratios=[1.0, 0.7], hspace=0.35, wspace=0.12
)

# Top row
ax_kw = fig2.add_subplot(gs2[0, 0])
ax_dmsgb = fig2.add_subplot(gs2[0, 1])
ax_dhyb = fig2.add_subplot(gs2[0, 2])

im_kw = ax_kw.imshow(_kw, extent=extent, aspect="auto", norm=s_norm, cmap="seismic")
im_dmsgb = ax_dmsgb.imshow(
    _d_kw_msgb, extent=extent, aspect="auto", norm=d_norm, cmap="RdBu_r"
)
im_dhyb = ax_dhyb.imshow(
    _d_kw_hyb, extent=extent, aspect="auto", norm=d_norm, cmap="RdBu_r"
)

ax_kw.set_title(r"$\max_{z_s} p_t^{\mathrm{k\text{-}Wave}}(y_s,t)$", fontsize=11)
ax_dmsgb.set_title(
    r"$\max_{z_s}(p_t^{\mathrm{k\text{-}Wave}} - p_t^{\mathrm{MSGB}})$", fontsize=11
)
ax_dhyb.set_title(
    r"$\max_{z_s}(p_t^{\mathrm{k\text{-}Wave}} - p_t^{\mathrm{Hybrid}})$", fontsize=11
)

for ax in (ax_kw, ax_dmsgb, ax_dhyb):
    ax.set_xlabel(r"$y_s$ (m)", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
ax_kw.set_ylabel("t (s)", fontsize=10)

# Colorbars
divL = make_axes_locatable(ax_kw)
caxL = divL.append_axes("left", size="5%", pad=0.15)
fig2.colorbar(im_kw, cax=caxL)
caxL.yaxis.set_ticks_position("left")
caxL.yaxis.set_label_position("left")

divR = make_axes_locatable(ax_dhyb)
caxR = divR.append_axes("right", size="5%", pad=0.15)
fig2.colorbar(im_dhyb, cax=caxR)

# Vertical profile line
for ax in (ax_kw, ax_dmsgb, ax_dhyb):
    ln = ax.axvline(y_pos, ls="--", lw=1.5, color="k", zorder=5)
    ln.set_path_effects([pe.Stroke(linewidth=3, foreground="white"), pe.Normal()])

# Bottom row: temporal profile at y_idx
ax_prof = fig2.add_subplot(gs2[1, :])
y_kw = _kw[:, y_idx]
y_msgb = _msgb[:, y_idx]
y_hyb = _hyb[:, y_idx]

ax_prof.plot(_ts, y_kw, label="k-Wave", lw=2, alpha=0.9)
ax_prof.plot(_ts, y_msgb, "--", label="MSGB", lw=2, alpha=0.9)
ax_prof.plot(_ts, y_hyb, ":", label="Hybrid", lw=2.5, alpha=0.9)

ax_prof.set_xlabel("Time (s)", fontsize=11)
ax_prof.set_ylabel(r"$\max_{z_s} p_t(y_s,t)$", fontsize=11)
ax_prof.set_title(f"Temporal profile at $y_s$ = {y_pos:.2e} m", fontsize=11)
ax_prof.legend(frameon=True, fancybox=True, shadow=True, loc="best")
ax_prof.grid(True, alpha=0.3, ls=":")

fig2.tight_layout()
fig2.savefig(PLOT_DIR / f"3d_fig2_sensor_{exp}.png", dpi=300, bbox_inches="tight")
plt.close(fig2)